# 🪨 Soil Grain-Size Distribution from Photos
### Physically-scaled DINOv2 embeddings + logit-target ridge, validated for camera shift

The task is to predict the **cumulative grain-size distribution** (sieve/hydrometer curve) of a soil sample at the 11 DIN EN ISO 14688-1 support points, **from photographs alone**.

**What makes this dataset tricky**

| Challenge | What we see in the data | How this notebook handles it |
|---|---|---|
| Very few labels | 24 training samples (127 photos), 10 test samples (35 photos) | Frozen pretrained encoders + heavily regularised ridge regression; no fine-tuning |
| **Camera shift** | Train photos come from Samsung A52 / Motorola phones, **all test photos come from iPhone 14 / 16** | Every image is resampled to the **same physical resolution (4 px/mm)** using the provided `ppm` calibration; models are selected with a **cross-camera** validation scheme |
| Resolution mismatch | Train images were downscaled (~4.5 px/mm) while test images are full resolution (14–19.5 px/mm) | Same physical rescaling as above, so a 2 mm grain covers the same number of pixels in every image |
| Metric on a log axis | Log-weighted Earth Mover's Distance (an **L1** loss) | Metric implemented exactly; regression in **logit space** (median-like, bounded predictions); post-processed to valid, monotone curves |

**Pipeline**

1. Parse files → sample IDs, attach per-phone calibration (pixels per mm).
2. Rescale every photo to **4 px/mm**, mask out the dark tray frame.
3. Extract features per photo:
   * **DINOv2 embeddings** (final model) – frozen self-supervised ViT-S/14 and ViT-B/14, applied to 56 mm and 112 mm soil tiles with flip test-time augmentation.
   * **Granulometry features** (interpretable reference) – morphological pattern spectra with discs of *physical* diameter 0.5–32 mm, band-pass energy, local contrast, colour.
4. Validate with **leave-one-sample-out** and **leave-one-sample-out × cross-camera** CV.
5. **Final model:** DINOv2-S and DINOv2-B on **56 mm tiles**, each with a ridge regression fitted on **logit-transformed** cumulative percentages; the two are averaged, predictions are averaged over all photos of a sample and post-processed into valid curves.

> **Leaderboard lessons (section 8).** (1) Hand-crafted granulometry looked best in CV but scored 104 on the public leaderboard, DINOv2 scored 49.6.
> (2) Models built on 112 mm tiles repeatedly scored worse than 56 mm tiles on the iPhone test photos.
> (3) Switching the regression target to logit space improved every model in CV by 4–6 EMD and the public score from 49.6 to **46.5**.

## 1. Setup

In [ ]:
import os, re, glob, time, unicodedata, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from PIL import Image, ImageOps

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
import logging; logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
SEED = 42
np.random.seed(SEED)
cv2.setNumThreads(4)
pd.set_option("display.precision", 2)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# ---- locate the competition data (Kaggle or local) ---------------------------------------
def find_data_root():
    if os.environ.get("SOIL_DATA_DIR"):
        return os.environ["SOIL_DATA_DIR"]
    hits = glob.glob("/kaggle/input/**/Training_labels_updated.csv", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError("Competition data not found - attach the dataset or set SOIL_DATA_DIR.")

DATA_ROOT = find_data_root()
def find_dir(pattern):
    cands = [d for d in glob.glob(os.path.join(DATA_ROOT, "**", pattern), recursive=True) if os.path.isdir(d)]
    # the image folders are nested twice (e.g. Test_All_Photos/Test_All_Photos) -> take the deepest one
    return max(cands, key=len)

TRAIN_DIR = find_dir("Training-All_Photos*")
TEST_DIR  = find_dir("Test_All_Photos*")
OUT_DIR   = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

print("train images:", len(os.listdir(TRAIN_DIR)), "| test images:", len(os.listdir(TEST_DIR)))

In [ ]:
# ---- configuration ------------------------------------------------------------------------
SIEVES = np.array([0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2.0, 6.3, 20.0, 63.0, 200.0])  # mm
TARGET_PPM = 4.0          # common physical resolution [px/mm] for every image
TILE_PX = 224             # DINOv2 tile size -> 224 px / 4 px/mm = 56 mm of soil
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)

# DINOv2 variants: (backbone, tile scale). Scale 1.0 -> 56 mm tiles, 0.5 -> 112 mm tiles.
if DEVICE == "cuda":
    DINO_VARIANTS = [("S", 1.0), ("B", 1.0), ("S", 0.5), ("B", 0.5)]   # 112 mm variants: comparison only
    USE_TTA = True
else:  # CPU fallback keeps the run time reasonable (enable a GPU for the full model)
    DINO_VARIANTS = [("S", 1.0)]
    USE_TTA = False
FINAL_SCALE = 1.0         # the final model uses the 56 mm tiles only (see section 8)
DINO_ALPHA = {"S": 1000, "B": 3000}   # ridge strength per backbone (384-d / 768-d, chosen by CV)
LOGIT_EPS = 0.5           # targets are clipped to [0.5, 99.5] % before the logit transform
print("device:", DEVICE, "| DINOv2 variants:", DINO_VARIANTS, "| flip TTA:", USE_TTA)

## 2. Evaluation metric

Log-weighted Earth Mover's Distance = the **area between the true and predicted cumulative curves on a log₁₀(d) axis**.
For each of the 10 intervals between consecutive support points the absolute error is integrated over the interval width
$\Delta_i = \log_{10}(x_{i+1}/x_i)$ (trapezoidal rule). The axis spans 5 decades, so the metric ranges from 0 to 500.

> A rectangle-rule variant ($\sum_i |F_i-\hat F_i|\,\Delta_i$) differs by < 2 % on this data and ranks models identically.

Submissions must be in [0, 100], non-decreasing, and equal 100 at 200 mm — `make_valid` enforces this.

In [ ]:
DLOG = np.diff(np.log10(SIEVES))

def emd_per_sample(y_true, y_pred):
    e = np.abs(np.asarray(y_true, float) - np.asarray(y_pred, float))
    return ((e[:, :-1] + e[:, 1:]) / 2 * DLOG).sum(axis=1)

def emd(y_true, y_pred):
    return float(emd_per_sample(y_true, y_pred).mean())

def make_valid(P):
    P = np.clip(np.asarray(P, float), 0, 100)
    P = np.maximum.accumulate(P, axis=1)   # cumulative curves are monotone
    P[:, -1] = 100.0                       # nothing in the dataset is larger than 200 mm
    return P

assert emd(np.zeros((1, 11)), np.zeros((1, 11))) == 0
assert np.isclose(emd(np.zeros((1, 11)), np.full((1, 11), 100.0)), 500)

## 3. Data loading and file → sample mapping

In [ ]:
labels = pd.read_csv(os.path.join(DATA_ROOT, "Training_labels_updated.csv"))
ppm_tab = pd.read_csv(os.path.join(DATA_ROOT, "ppm_updated.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))
TARGET_COLS = list(labels.columns[1:])
display(ppm_tab)
display(labels.head())

In [ ]:
# filename prefix -> phone (row of ppm_updated.csv). Longest prefix first.
PHONE_PATTERNS = [
    (r"^Motorola_Edge_60_fusion_", "Motorola Edge 60 Fusion"),
    (r"^Motorola_Edge_",           "Motorola Edge"),
    (r"^Samsung_A52_",             "Samsung A52"),
    (r"^iPhone_?14_",              "iPhone 14"),
    (r"^iPhone_?16_",              "iPhone 16"),
]

def to_sample_id(raw):
    s = unicodedata.normalize("NFC", raw)
    for a, b in {"ü": "ue", "ö": "oe", "ä": "ae", "ß": "ss", ",": "_"}.items():
        s = s.replace(a, b)
    return s.strip()

def parse_filename(fname):
    stem = os.path.splitext(unicodedata.normalize("NFC", fname))[0]
    for pat, phone in PHONE_PATTERNS:
        if re.match(pat, stem):
            rest = re.sub(pat, "", stem)
            break
    else:
        raise ValueError(f"unknown phone prefix: {fname}")
    rest = re.sub(r"_\d+$", "", rest)         # training:  ..._H038_01.jpg
    rest = re.sub(r"\s*\(\d+\)$", "", rest)   # test:      ... (2).JPG
    return phone, to_sample_id(rest)

def id_key(s):
    # accent/encoding-insensitive key: "Münster_BS6_9,0-10m" == "Muenster_BS6_9_0-10m"
    s = unicodedata.normalize("NFKD", unicodedata.normalize("NFC", s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]", "", s.replace("ue", "u").replace("oe", "o").replace("ae", "a"))

KNOWN_IDS = {"train": {id_key(s): s for s in labels.sample_id}, "test": {id_key(s): s for s in sample_sub.sample_id}}

rows = []
for split, d in [("train", TRAIN_DIR), ("test", TEST_DIR)]:
    for f in sorted(os.listdir(d)):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            phone, sid = parse_filename(f)
            sid = KNOWN_IDS[split].get(id_key(sid), sid)   # snap to the official ID spelling
            rows.append(dict(split=split, file=f, path=os.path.join(d, f), phone=phone, sample_id=sid))
images = pd.DataFrame(rows)

# sanity checks: every image maps to a known sample and every sample has images
assert set(images.query("split=='train'").sample_id) == set(labels.sample_id)
assert set(images.query("split=='test'").sample_id) == set(sample_sub.sample_id)
print(images.groupby("split").agg(photos=("file", "size"), samples=("sample_id", "nunique")))

## 4. Exploratory analysis
### 4.1 Grading curves of the training samples

In [ ]:
def setup_log_axis(ax):
    ax.set_xscale("log"); ax.set_xlim(SIEVES[0], SIEVES[-1]); ax.set_ylim(0, 101)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    for b, name in [(0.002, "clay"), (0.063, "silt"), (2, "sand"), (63, "gravel"), (200, "cobble")]:
        ax.axvline(b, color="grey", lw=0.6, ls=":")
    ax.set_xlabel("grain diameter [mm]"); ax.set_ylabel("cumulative passing [%]")

Ytrain = labels[TARGET_COLS].values
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={"width_ratios": [1.15, 1]})
cmap = plt.cm.viridis(np.linspace(0, 1, len(labels)))
order = np.argsort(Ytrain[:, 6])  # sort by % < 2 mm (sand vs gravel)
for c, i in zip(cmap, order):
    axes[0].plot(SIEVES, Ytrain[i], color=c, lw=1.6, alpha=0.85)
axes[0].plot(SIEVES, Ytrain.mean(0), "k--", lw=2.5, label="training mean")
axes[0].plot(SIEVES, np.linspace(100/11, 100, 11), "r:", lw=2, label="equal-share baseline")
setup_log_axis(axes[0]); axes[0].legend(); axes[0].set_title("24 training grading curves")

# soil fractions according to DIN EN ISO 14688-1
frac = pd.DataFrame({
    "clay  <0.002":       Ytrain[:, 0],
    "silt  0.002–0.063":  Ytrain[:, 3] - Ytrain[:, 0],
    "sand  0.063–2":      Ytrain[:, 6] - Ytrain[:, 3],
    "gravel 2–63":        Ytrain[:, 9] - Ytrain[:, 6],
    "cobbles 63–200":     Ytrain[:, 10] - Ytrain[:, 9],
}, index=labels.sample_id).iloc[order]
frac.plot.bar(stacked=True, ax=axes[1], width=0.85, colormap="BrBG_r")
axes[1].set_ylabel("mass fraction [%]"); axes[1].set_title("Soil fractions per sample"); axes[1].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

The training set covers fine-grained loess-like soils (H405, F827: > 75 % fines) through well-graded sandy gravels (H366–H374).
Most curves pass through the **sand/gravel range (0.2–20 mm)** where they spread the most — this is where most of the metric is won or lost.

### 4.2 Cameras and physical resolution — the main source of train/test shift

In [ ]:
def image_size(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im)
        return im.size  # (w, h)

images[["w_px", "h_px"]] = np.array([image_size(p) for p in images.path])
ppm_lookup = ppm_tab.set_index("phone")
def effective_ppm(r):
    cal = ppm_lookup.loc[r.phone]
    # ppm is given for the native sensor size; scale it to the stored (possibly downscaled) image
    return cal.ppm * max(r.w_px, r.h_px) / max(cal.width, cal.height)
images["px_per_mm"] = images.apply(effective_ppm, axis=1)
images["field_of_view_mm"] = (images[["w_px", "h_px"]].max(axis=1) / images.px_per_mm).round(0)

summary = (images.groupby(["split", "phone"])
           .agg(photos=("file", "size"), samples=("sample_id", "nunique"),
                long_side_px=("w_px", lambda s: int(np.maximum(s, images.loc[s.index, "h_px"]).max())),
                px_per_mm=("px_per_mm", "mean"), long_side_mm=("field_of_view_mm", "mean")))
display(summary)

**Key observation.** Training photos were stored downscaled (~4.5 px/mm) while test photos are full-resolution iPhone images (14–19.5 px/mm),
and **no iPhone appears in training**. Feeding raw pixels to a model would make the same grain look 3–4× larger in the test set.
We therefore resample every image to **4 px/mm** so that texture scales correspond to the same physical grain sizes everywhere.

## 5. Preprocessing: physical rescaling and tray masking

In [ ]:
def load_physical(path, px_per_mm, target_ppm=TARGET_PPM):
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    a = np.asarray(im)
    s = target_ppm / px_per_mm
    return cv2.resize(a, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)

def soil_mask(rgb):
    # the sample tray is black: keep pixels that are not dark, then clean small specks
    v = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[..., 2]
    m = (v > 60).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((15, 15), np.uint8))
    return m.astype(bool)

def center_crop(a, size):
    h, w = a.shape[:2]; y, x = (h - size) // 2, (w - size) // 2
    return a[y:y + size, x:x + size]

# same 80 x 80 mm window from different cameras / soils
show = [("train", "Samsung A52", "H038"), ("train", "Motorola Edge", "H405"), ("train", "Samsung A52", "H616"),
        ("test", "iPhone 16", "HPC_Airbus BS6-3"), ("test", "iPhone 14", "HPC_Testfeld Lidl WHV")]
fig, axes = plt.subplots(1, len(show), figsize=(18, 4))
for ax, (sp, ph, sid) in zip(axes, show):
    r = images.query("split==@sp and phone==@ph and sample_id==@sid").iloc[0]
    rgb = load_physical(r.path, r.px_per_mm)
    ax.imshow(center_crop(rgb, int(80 * TARGET_PPM))); ax.axis("off")
    ax.set_title(f"{sp}: {sid}\n{ph}", fontsize=9)
plt.suptitle("80 × 80 mm crops after rescaling to 4 px/mm (same physical scale)", y=1.02); plt.show()

## 6. Feature extraction
### 6.1 Granulometry features (physically scaled, hand-crafted)

On a grey-level image (normalised per photo to zero mean / unit variance, so exposure and white balance matter less):

* **Morphological pattern spectrum** – `open_d`: brightness removed by an opening with a disc of diameter *d* mm (bright grains smaller than *d* vanish);
  `close_d`: same for dark gaps/shadows between grains. *d* ∈ {0.5, 1, 2, 4, 8, 16, 32} mm. This is a classic image-based sieve.
* **Band-pass energy** – `dog_s`: log-energy of the difference between Gaussian blurs at successive physical scales *s* ∈ {0.25 … 16} mm.
* **Local contrast** – `lstd_w`: mean local standard deviation in windows of 1, 4, 16 mm; gradient statistics.
* **Colour** – Lab statistics (mineralogy / moisture / fines content cues).

In [ ]:
GRANULO_DIAMS = [0.5, 1, 2, 4, 8, 16, 32]   # mm
DOG_SCALES = [0.25, 0.5, 1, 2, 4, 8, 16]    # mm
LSTD_WINDOWS = [1, 4, 16]                   # mm

def granulometry_features(rgb, mask):
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    m = cv2.erode(mask.astype(np.uint8), np.ones((41, 41), np.uint8)).astype(bool)  # stay away from the frame
    if m.mean() < 0.2:
        m = mask
    gn = (g - g[m].mean()) / (g[m].std() + 1e-6)
    f = {}
    for d in GRANULO_DIAMS:
        ds = 1 if d < 8 else (2 if d < 32 else 4)          # large discs on a downsampled image (speed)
        G = gn if ds == 1 else cv2.resize(gn, None, fx=1/ds, fy=1/ds, interpolation=cv2.INTER_AREA)
        M = m if ds == 1 else cv2.resize(m.astype(np.uint8), (G.shape[1], G.shape[0]),
                                         interpolation=cv2.INTER_NEAREST).astype(bool)
        r = max(1, int(round(d * TARGET_PPM / ds / 2)))
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
        f[f"open_{d}"] = float((G - cv2.morphologyEx(G, cv2.MORPH_OPEN, k))[M].mean())
        f[f"close_{d}"] = float((cv2.morphologyEx(G, cv2.MORPH_CLOSE, k) - G)[M].mean())
    prev = gn
    for s in DOG_SCALES:
        b = cv2.GaussianBlur(gn, (0, 0), s * TARGET_PPM)
        f[f"dog_{s}"] = float(np.log(((prev - b)[m] ** 2).mean() + 1e-6))
        prev = b
    for w in LSTD_WINDOWS:
        k = max(3, int(w * TARGET_PPM) | 1)
        mu, mu2 = cv2.blur(gn, (k, k)), cv2.blur(gn * gn, (k, k))
        f[f"lstd_{w}"] = float(np.sqrt(np.clip(mu2 - mu * mu, 0, None))[m].mean())
    gx, gy = cv2.Sobel(gn, cv2.CV_32F, 1, 0), cv2.Sobel(gn, cv2.CV_32F, 0, 1)
    mag = np.sqrt(gx ** 2 + gy ** 2)[m]
    f["grad_mean"], f["grad_p90"] = float(mag.mean()), float(np.percentile(mag, 90))
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    for i, ch in enumerate("Lab"):
        f[f"{ch}_std"] = float(lab[..., i][m].std())
    f["a_mean"], f["b_mean"] = float(lab[..., 1][m].mean()), float(lab[..., 2][m].mean())
    return f

### 6.2 DINOv2 tile embeddings (final model features)

Frozen, self-supervised **DINOv2** encoders (ViT-S/14 and ViT-B/14) embed 224-px soil tiles cut from the 4 px/mm image (**56 mm** tiles)
and from a 2 px/mm copy (**112 mm** tiles), with 50 % overlap; tiles touching the tray frame are skipped.
Each tile is embedded as-is and flipped/transposed (test-time augmentation), and all embeddings are averaged per photo.
Because the encoders are frozen and the downstream model is a heavily regularised ridge, 24 samples are enough.

*Requires **Internet = On** (timm downloads the weights from the Hugging Face hub).*

In [ ]:
import timm, io, contextlib
BACKBONES = {"S": "vit_small_patch14_dinov2.lvd142m", "B": "vit_base_patch14_dinov2.lvd142m"}
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
encoders = {}
for key in sorted({b for b, _ in DINO_VARIANTS}):
    try:
        with contextlib.redirect_stderr(io.StringIO()):   # silence hub download chatter
            encoders[key] = timm.create_model(BACKBONES[key], pretrained=True, num_classes=0,
                                              img_size=TILE_PX).to(DEVICE, DTYPE).eval()
    except Exception as e:
        raise RuntimeError("Could not download DINOv2 weights - turn on Internet in the notebook settings.") from e
    print(f"DINOv2-{key} loaded: {sum(p.numel() for p in encoders[key].parameters()) / 1e6:.1f} M params")
_mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
_std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def soil_tiles(rgb, mask, size=TILE_PX, stride=TILE_PX // 2):
    H, W = mask.shape
    for max_dark in (0.05, 0.25, 1.01):            # relax if the photo shows a lot of frame
        out = [rgb[y:y + size, x:x + size]
               for y in range(0, H - size + 1, stride) for x in range(0, W - size + 1, stride)
               if 1 - mask[y:y + size, x:x + size].mean() <= max_dark]
        if len(out) >= 4:
            return out
    return out or [cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)]

@torch.no_grad()
def embed_tiles(model, tiles):
    x = torch.from_numpy(np.stack(tiles)).to(DEVICE).permute(0, 3, 1, 2).float().div(255)
    x = ((x - _mean) / _std).to(DTYPE)
    views = [x, x.flip(3), x.flip(2), x.transpose(2, 3)] if USE_TTA else [x]
    embs = [torch.cat([model(v[i:i + 64]) for i in range(0, len(v), 64)]).float().mean(0) for v in views]
    return torch.stack(embs).mean(0).cpu().numpy()

def variant_name(backbone, scale):
    return f"DINOv2-{backbone} · {int(round(TILE_PX / TARGET_PPM / scale))} mm tiles"

def dino_embeddings(rgb, mask):
    out = {}
    for scale in sorted({s for _, s in DINO_VARIANTS}, reverse=True):
        if scale == 1.0:
            img, m = rgb, mask
        else:
            img = cv2.resize(rgb, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
            m = cv2.resize(mask.astype(np.uint8), (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
        tiles = soil_tiles(img, m)
        for b, s in DINO_VARIANTS:
            if s == scale:
                out[variant_name(b, s)] = embed_tiles(encoders[b], tiles)
    return out

### 6.3 Run extraction for all photos

In [ ]:
t0 = time.time()
hc_rows, dino_rows = [], []
for i, r in enumerate(images.itertuples()):
    rgb = load_physical(r.path, r.px_per_mm)
    mask = soil_mask(rgb)
    hc_rows.append(granulometry_features(rgb, mask))
    dino_rows.append(dino_embeddings(rgb, mask))
    if (i + 1) % 25 == 0 or i + 1 == len(images):
        print(f"{i + 1:4d}/{len(images)} photos  ({time.time() - t0:5.0f}s)")

HC = pd.DataFrame(hc_rows)
HC_COLS = list(HC.columns)
X_hc = HC.values.astype(np.float64)
DINO_NAMES = [variant_name(b, s) for b, s in DINO_VARIANTS]
DINO_META = {variant_name(b, s): (b, s) for b, s in DINO_VARIANTS}
X_dino = {n: np.stack([r[n] for r in dino_rows]).astype(np.float64) for n in DINO_NAMES}
print("granulometry features:", X_hc.shape)
for n, X in X_dino.items():
    print(f"{n}: {X.shape}")

In [ ]:
# How do the granulometry features relate to the targets? (photo-level, training only)
is_tr = (images.split == "train").values
Y_img = labels.set_index("sample_id").loc[images.loc[is_tr, "sample_id"], TARGET_COLS].values
corr = pd.DataFrame({t: [np.corrcoef(X_hc[is_tr, j], Y_img[:, k])[0, 1] for j in range(len(HC_COLS))]
                     for k, t in enumerate(TARGET_COLS[:-1])}, index=HC_COLS)
fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr.columns)), [f"F({c} mm)" for c in corr.columns], rotation=45, ha="right")
ax.set_yticks(range(len(HC_COLS)), HC_COLS, fontsize=8); ax.grid(False)
plt.colorbar(im, label="Pearson r"); ax.set_title("Granulometry features vs. % passing at each sieve")
plt.tight_layout(); plt.show()

In [ ]:
# Where do the test photos sit relative to the training photos in feature space?
Z = StandardScaler().fit(X_hc[is_tr]).transform(X_hc)
pc = PCA(2, random_state=SEED).fit(Z[is_tr]).transform(Z)
fig, ax = plt.subplots(figsize=(8, 5.5))
for ph, mk in [("Samsung A52", "o"), ("Motorola Edge", "s"), ("Motorola Edge 60 Fusion", "D"), ("iPhone 14", "^"), ("iPhone 16", "v")]:
    sel = (images.phone == ph).values
    ax.scatter(pc[sel, 0], pc[sel, 1], marker=mk, s=45, alpha=0.85, edgecolor="k", lw=0.4,
                    label=f"{ph} ({'test' if not is_tr[sel].any() else 'train'})")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.legend(fontsize=8)
ax.set_title("Granulometry feature space (PCA fit on training photos)"); plt.show()

Most test photos fall within the cloud spanned by the training photos; a few (coarse gravels) extend it.
To keep linear models from extrapolating wildly there, features are **clipped to the training range widened by 50 %** before scaling (`TrainRangeClipper` below) —
strict clipping hurts under camera shift, no clipping leaves the model unguarded.

## 7. Validation strategy

Photos of the same sample are strongly correlated, so all CV is **grouped by sample**. Two schemes:

* **LOSO** – leave-one-sample-out: train on 23 samples, predict the held-out sample (all its photos averaged).
* **Cross-camera LOSO** – additionally, the model is trained **only on photos from other phones** than the held-out photos.
  This mimics the real test situation (unseen iPhone camera) and is our main model-selection criterion together with LOSO.

In [ ]:
class TrainRangeClipper(BaseEstimator, TransformerMixin):
    # Clip each feature to the training range widened by `margin` x range on both sides:
    # moderate extrapolation is allowed (new cameras shift features a bit), wild extrapolation is not.
    # margin=0.5 was chosen on the cross-camera CV (0 -> too strict, None -> unguarded).
    def __init__(self, margin=0.5):
        self.margin = margin
    def fit(self, X, y=None):
        lo, hi = X.min(0), X.max(0)
        self.lo_, self.hi_ = lo - self.margin * (hi - lo), hi + self.margin * (hi - lo); return self
    def transform(self, X):
        return np.clip(X, self.lo_, self.hi_)

class MeanCurve(BaseEstimator):
    def fit(self, X, y): self.m_ = y.mean(0); return self
    def predict(self, X): return np.tile(self.m_, (len(X), 1))

class KNNCurve(BaseEstimator):
    # distance-weighted k-NN in standardised PCA space (included to show its camera sensitivity)
    def __init__(self, k=5, n_pca=10): self.k, self.n_pca = k, n_pca
    def fit(self, X, y):
        self.t_ = make_pipeline(StandardScaler(), PCA(min(self.n_pca, X.shape[1]), random_state=SEED)).fit(X)
        self.Z_, self.y_ = self.t_.transform(X), y; return self
    def predict(self, X):
        d = ((self.t_.transform(X)[:, None] - self.Z_[None]) ** 2).sum(-1) ** 0.5
        idx = np.argsort(d, 1)[:, :self.k]; w = 1 / (np.take_along_axis(d, idx, 1) + 1e-6)
        return (self.y_[idx] * w[..., None]).sum(1) / w.sum(1, keepdims=True)

class LogitRidge(BaseEstimator):
    # Ridge regression on logit-transformed cumulative percentages.
    # The EMD is an L1 loss, so the best prediction is a *median*-like curve, not a mean. Fitting in logit space and
    # transforming back pulls predictions towards the typical shape (e.g. near-zero clay for sands) instead of the
    # mean of a skewed distribution, and keeps every prediction inside (0, 100). The 200 mm point is always 100.
    def __init__(self, alpha=1000, eps=LOGIT_EPS): self.alpha, self.eps = alpha, eps
    def fit(self, X, y):
        p = np.clip(y[:, :-1], self.eps, 100 - self.eps) / 100
        self.m_ = make_pipeline(StandardScaler(), Ridge(alpha=self.alpha)).fit(X, np.log(p / (1 - p))); return self
    def predict(self, X):
        p = 100 / (1 + np.exp(-self.m_.predict(X)))
        return np.hstack([p, np.full((len(p), 1), 100.0)])

groups = images.loc[is_tr, "sample_id"].values
phones = images.loc[is_tr, "phone"].values
Y_lab = labels.set_index("sample_id")[TARGET_COLS]

def build_folds():
    loso, xcam = [], []
    for s in labels.sample_id:
        loso.append((s, groups != s, groups == s))
        for p in np.unique(phones[groups == s]):
            tr = (groups != s) & (phones != p)
            if tr.sum() >= 10:
                xcam.append((s, tr, (groups == s) & (phones == p)))
    return {"LOSO": loso, "Cross-camera": xcam}
FOLDS = build_folds()
print({k: len(v) for k, v in FOLDS.items()}, "folds")

def cv_predict(make_model, X):
    out = {}
    for scheme, folds in FOLDS.items():
        P = []
        for s, tr, te in folds:
            mdl = make_model().fit(X[tr], Y_img[tr])
            P.append(make_valid(mdl.predict(X[te])).mean(0))   # average over the sample's photos
        out[scheme] = np.array(P)
    return out

Y_true = {k: Y_lab.loc[[s for s, _, _ in v]].values for k, v in FOLDS.items()}

In [ ]:
FEATS_ALL = {"hc": X_hc, **X_dino}                 # all photos (train + test)
FEATS = {k: v[is_tr] for k, v in FEATS_ALL.items()}  # training photos only

MODELS = {
    "Training mean":          (lambda: MeanCurve(), "hc"),
    "Granulometry · Ridge":   (lambda: make_pipeline(TrainRangeClipper(), StandardScaler(), Ridge(alpha=10)), "hc"),
    "Granulometry · kNN(5)":  (lambda: KNNCurve(5), "hc"),
}
for n in DINO_NAMES:
    a = DINO_ALPHA[DINO_META[n][0]]
    MODELS[n + " · Ridge"] = (lambda: make_pipeline(StandardScaler(), Ridge(alpha=1000)), n)
    MODELS[n + " · Logit ridge"] = (lambda a=a: LogitRidge(alpha=a), n)
OOF, rows = {}, []
for name, (mk, fs) in MODELS.items():
    OOF[name] = cv_predict(mk, FEATS[fs])
    rows.append({"model": name, **{k: emd(Y_true[k], OOF[name][k]) for k in FOLDS}})
baseline_curve = np.round(np.arange(1, 12) * 100 / 11, 2)
rows.insert(0, {"model": "Equal-share baseline", **{k: emd(Y_true[k], np.tile(baseline_curve, (len(Y_true[k]), 1))) for k in FOLDS}})
cv_table = pd.DataFrame(rows).set_index("model")
cv_table["mean"] = cv_table.mean(1)
display(cv_table.style.format("{:.2f}").background_gradient(cmap="RdYlGn_r", axis=0))

* All image models cut the error of the trivial baseline by more than half in CV.
* **Logit ridge beats plain ridge for every DINOv2 variant**, in both CV schemes — the EMD rewards median-like predictions.
* **k-NN collapses under camera shift** (it memorises camera-specific feature offsets) — a good reminder why the cross-camera CV matters.
* The granulometry ridge looks competitive here, but CV only covers the training soils and phones — see the leaderboard check below.

## 8. Leaderboard check and final model

Cross-validation can only measure generalisation to the **training soils and Android phones**. The test soils (northern-German sites, all photographed with iPhones
at 3–4× the training resolution) are a different population, so model families were checked with a few public-leaderboard submissions
(public LB = ~30 % of the test samples, i.e. only ~3 samples — used to choose *families*, never to tune weights):

| Submission | Tiles | Target | CV cross-camera | Public EMD |
|---|---|---|---|---|
| Equal-share baseline | – | – | – | 125.6 |
| Training-mean curve | – | – | – | 102.4 |
| Granulometry ridge + DINOv2 blend (best in early CV) | – | raw | – | 104.4 |
| DINOv2-S + B ensemble | 56 + 112 mm | raw | 50.4 | 62.0 |
| DINOv2-B + L ensemble | 112 mm | logit | 42.7 | 79.9 |
| DINOv2-S ridge | 56 mm | raw | 53.3 | 49.6 |
| **DINOv2-S + B (final)** | **56 mm** | **logit** | **47.1** | **46.5** |

Two consistent findings:
1. **Hand-crafted statistics do not transfer** — at 0.25 mm/pixel grey sand looks "smooth" and is read as silt.
2. **112 mm tiles do not transfer to the iPhone photos.** They are best in CV (where every photo comes from a ~4.5 px/mm Android phone) but every submission that
   contained them was worse on the test photos. The 56 mm tiles keep more of the grain detail that the sharp iPhone images carry. CV cannot see this shift, the leaderboard can.

Among the 56 mm models, the **logit target** is better both in CV (47.1 vs 53.3) and on the leaderboard (46.5 vs 49.6), so the two checks agree.
**Final model:** equal-weight average of the DINOv2-S and DINOv2-B logit ridges on 56 mm tiles.

In [ ]:
BLEND_MEMBERS = [n + " · Logit ridge" for n in DINO_NAMES if DINO_META[n][1] == FINAL_SCALE]
WEIGHTS = np.full(len(BLEND_MEMBERS), 1 / len(BLEND_MEMBERS))

def blend(ws, scheme):
    return make_valid(sum(w * OOF[m][scheme] for w, m in zip(ws, BLEND_MEMBERS)))

print("final model:", " + ".join(BLEND_MEMBERS))
for k in FOLDS:
    print(f"{k:>13s} EMD  ensemble = {emd(Y_true[k], blend(WEIGHTS, k)):6.2f}   "
          f"(best member = {cv_table.loc[BLEND_MEMBERS, k].min():6.2f}, baseline = {cv_table.loc['Equal-share baseline', k]:6.2f})")

### 8.1 Out-of-fold diagnostics (LOSO)

In [ ]:
P_oof = blend(WEIGHTS, "LOSO")
ids_loso = [s for s, _, _ in FOLDS["LOSO"]]
err = pd.Series(emd_per_sample(Y_true["LOSO"], P_oof), index=ids_loso).sort_values()

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.bar(err.index, err.values, color=plt.cm.RdYlGn_r(err.values / err.max()))
ax.axhline(err.mean(), color="k", ls="--", label=f"mean = {err.mean():.1f}")
ax.set_ylabel("EMD"); ax.set_title("Per-sample LOSO error of the final model"); ax.legend(); plt.xticks(rotation=60); plt.show()

show_ids = list(err.index[:3]) + list(err.index[len(err)//2 - 1:len(err)//2 + 2]) + list(err.index[-3:])
fig, axes = plt.subplots(3, 3, figsize=(15, 10.5), sharex=True, sharey=True)
for ax, s in zip(axes.ravel(), show_ids):
    i = ids_loso.index(s)
    ax.plot(SIEVES, Y_true["LOSO"][i], "o-", color="k", label="laboratory")
    ax.plot(SIEVES, P_oof[i], "s--", color="tab:orange", label="predicted (OOF)")
    ax.fill_between(SIEVES, Y_true["LOSO"][i], P_oof[i], color="tab:orange", alpha=0.15)
    setup_log_axis(ax); ax.set_title(f"{s}  —  EMD {err[s]:.1f}", fontsize=10)
axes[0, 0].legend(); plt.suptitle("Best / median / worst out-of-fold predictions", y=1.0); plt.tight_layout(); plt.show()

The largest errors come from fine-grained soils that form **aggregates** when dry (e.g. H405): the surface looks gravelly
while the laboratory curve (after dispersion) is mostly silt/clay. This is a fundamental limitation of surface photographs.

## 9. Final training and test prediction

In [ ]:
is_te = ~is_tr
P_test_img = 0
for w, name in zip(WEIGHTS, BLEND_MEMBERS):
    mk, fs = MODELS[name]
    P_test_img = P_test_img + w * make_valid(mk().fit(FEATS_ALL[fs][is_tr], Y_img).predict(FEATS_ALL[fs][is_te]))
P_test_img = make_valid(P_test_img)

test_imgs = images.loc[is_te, ["sample_id", "file", "phone"]].reset_index(drop=True)
pred_img = pd.concat([test_imgs, pd.DataFrame(P_test_img, columns=TARGET_COLS)], axis=1)

# aggregate photos -> sample (mean of valid curves is a valid curve)
pred_sample = pred_img.groupby("sample_id")[TARGET_COLS].mean()
submission = sample_sub[["sample_id"]].merge(pred_sample, left_on="sample_id", right_index=True, how="left")
submission[TARGET_COLS] = make_valid(submission[TARGET_COLS].values).round(4)
display(submission)

In [ ]:
# photo-to-photo agreement within each test sample (a proxy for prediction stability)
spread = pred_img.groupby("sample_id").apply(
    lambda g: emd_per_sample(g[TARGET_COLS].values, np.tile(g[TARGET_COLS].values.mean(0), (len(g), 1))).mean())
print("mean within-sample photo disagreement (EMD to sample mean):")
display(spread.rename("EMD").to_frame().T)

fig, ax = plt.subplots(figsize=(10, 5.5))
for c, (_, r) in zip(plt.cm.tab10(np.arange(len(submission))), submission.iterrows()):
    ax.plot(SIEVES, r[TARGET_COLS].values.astype(float), "o-", color=c, lw=1.8, ms=4, label=r.sample_id)
ax.plot(SIEVES, Ytrain.mean(0), "k--", lw=2, label="training mean")
setup_log_axis(ax); ax.legend(fontsize=8, ncol=2); ax.set_title("Predicted grading curves for the test samples"); plt.show()

## 10. Submission checks and export

In [ ]:
def check_submission(sub, template):
    vals = sub[TARGET_COLS].values
    assert list(sub.columns) == list(template.columns), "column mismatch"
    assert sub.sample_id.tolist() == template.sample_id.tolist(), "row order / ids mismatch"
    assert np.isfinite(vals).all(), "NaN / inf in predictions"
    assert (vals >= 0).all() and (vals <= 100).all(), "values outside [0, 100]"
    assert (np.diff(vals, axis=1) >= 0).all(), "curve not non-decreasing"
    assert np.allclose(vals[:, -1], 100), "value at 200 mm must be 100"
    return True

check_submission(submission, sample_sub)
path = os.path.join(OUT_DIR, "submission.csv")
submission.to_csv(path, index=False)
print("✅ submission passes all structural checks ->", path)
print(open(path).read())

In [ ]:
final = pd.DataFrame({
    "Equal-share baseline": cv_table.loc["Equal-share baseline", list(FOLDS)],
    "Training mean":        cv_table.loc["Training mean", list(FOLDS)],
    "Granulometry ridge":   cv_table.loc["Granulometry · Ridge", list(FOLDS)],
    "DINOv2-S · 56 mm · plain ridge": cv_table.loc[DINO_NAMES[0] + " · Ridge", list(FOLDS)],
    "Final: DINOv2-S+B · 56 mm · logit ridge": pd.Series({k: emd(Y_true[k], blend(WEIGHTS, k)) for k in FOLDS}),
}).T
display(final.style.format("{:.2f}").set_caption("Cross-validated EMD (lower is better)"))

## 11. Summary

**What worked**
* Calibrating every image to a common **physical resolution** (px/mm), so tile sizes mean the same thing in mm for every camera.
* Frozen **DINOv2** embeddings of **56 mm** soil tiles (ViT-S and ViT-B, flip TTA) + strongly regularised ridge.
* **Regressing in logit space**: the EMD is an L1 loss, so median-like predictions win; −4 to −6 EMD in CV for every variant, public 49.6 → 46.5.
* A **cross-camera** CV — k-NN looks fine in plain LOSO but fails when the camera changes.

**What did not transfer**
* Hand-crafted granulometry/colour statistics: competitive in CV (same soils, Android phones) but 104 on the public leaderboard.
* 112 mm tiles and DINOv2-L: best in CV, but clearly worse on the high-resolution iPhone test photos (62–80 vs 46–50).
  CV on 24 samples from one population and two phones cannot detect these shifts — an outside check was essential.

**Limitations / ideas**
* Clay and silt (< 0.063 mm) are far below the pixel size (0.25 mm); they are inferred indirectly from colour and aggregate texture.
* Dry fine-grained soils form aggregates that look like gravel — a fundamentally ambiguous case for surface photos.
* With more labelled data, fine-tuning a CNN on physically scaled tiles with a monotone (softmax-over-bins) head would be the natural next step.